# 06 - Structured degradation pilot (training data only)

CPU only. Run after notebook 05, on the same runtime. We freeze a deterministic simulator and inspect two training fingers per source database. No validation/test images, score thresholds or trained models are used.

Dry-like: local brightening and ridge erosion. Wet-like: local darkening and ridge thickening/valley filling. Generic: foreground blur, contrast reduction and noise. These are image-domain heuristics, NOT measured moisture or a validated physical sensor model. They do not establish performance on real dry/wet fingers.

Original impression 2 is the restoration reference, NOT assumed perfectly clean. Impression 1 of the same training finger is the independent gallery. Only the probe is degraded. Three levels per degradation family, with image-specific fixed seeds. An intervention map indicates where the simulator applied changes; it is NOT calibrated confidence, a dry/wet annotation or a minutiae reliability label.

We store a reusable simulator Python module on Drive so later training and evaluation use the exact same implementation. No quality-based replacement of pilot fingers. Missing/failed samples must be reviewed, not silently substituted. Matching against the independent gallery is a small train diagnostic, not an accuracy estimate or a test result.

Inspect the preview and score degradation before accepting this design for training. If the simulator is revised, create a new protocol version; never silently overwrite the frozen source.

In [ ]:
import hashlib
import importlib.util
import json
import re
import subprocess
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
PAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'
SIM_DIR = PROJECT_ROOT / 'protocols' / 'structured_degradation_v1'
NBIS_ROOT = LOCAL_ROOT / 'tools' / 'nbis_5_0_0'
MINDTCT = NBIS_ROOT / 'install' / 'bin' / 'mindtct'
BOZORTH3 = NBIS_ROOT / 'install' / 'bin' / 'bozorth3'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
WORK_DIR = LOCAL_ROOT / 'degradation_pilot' / RUN_ID
REPORT_DIR = PROJECT_ROOT / 'datasets' / 'structured_degradation_pilot_v1' / RUN_ID
for directory in (WORK_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True) + '\n').encode('utf-8')

split_protocol = json.loads((SPLIT_DIR / 'protocol.json').read_text(encoding='utf-8'))
for name, expected in split_protocol['file_sha256'].items():
    if file_digest(SPLIT_DIR / name) != expected:
        raise RuntimeError(f'Frozen split file changed: {name}')
locked = pd.read_csv(SPLIT_DIR / 'manifest_splits.csv', keep_default_na=False)
normalization = json.loads((PAIR_DIR / 'configuration.json').read_text(encoding='utf-8'))
if normalization['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json'):
    raise RuntimeError('Baseline and simulator reference different split protocols.')
if file_digest(PAIR_DIR / 'pairs.csv') != normalization['pair_csv_sha256']:
    raise RuntimeError('Frozen baseline pair list changed.')
if normalization['pillow_version'] != Image.__version__:
    raise RuntimeError('Pillow version differs from the frozen normalization protocol.')
build = json.loads((NBIS_ROOT / 'verified_build.json').read_text(encoding='utf-8'))
if build['signature']['source_sha256'] != normalization['nbis_source_sha256']:
    raise RuntimeError('NBIS source differs from the baseline protocol.')
for binary in (MINDTCT, BOZORTH3):
    if not binary.is_file() or file_digest(binary) != build['binary_sha256'][binary.name]:
        raise RuntimeError('NBIS runtime missing or changed; rerun notebook 04.')
print('Verified protocols. Pilot will use train only.')

In [ ]:
SIMULATOR_SOURCE = r'''"""Deterministic image-domain degradation heuristics, not physical moisture labels."""
import hashlib
import numpy as np
from PIL import Image, ImageFilter

VERSION = 'structured_degradation_v1'
SETTINGS = {
    'mild': {'coverage': 0.15, 'strength': 0.40, 'window': 3, 'blur': 0.5, 'noise': 0.015, 'contrast': 0.90},
    'moderate': {'coverage': 0.30, 'strength': 0.65, 'window': 3, 'blur': 0.9, 'noise': 0.030, 'contrast': 0.75},
    'severe': {'coverage': 0.45, 'strength': 0.85, 'window': 5, 'blur': 1.3, 'noise': 0.050, 'contrast': 0.60},
}

def validate_image(image):
    x = np.asarray(image, dtype=np.float32)
    if x.ndim != 2 or min(x.shape) < 16 or not np.isfinite(x).all():
        raise ValueError('Expected finite 2D grayscale image, dimensions >= 16.')
    if x.min() < 0 or x.max() > 1:
        raise ValueError('Expected image intensities in [0,1].')
    return x

def to_pil(image):
    x = validate_image(image)
    return Image.fromarray(np.rint(x * 255).astype(np.uint8))

def foreground_mask(image):
    x = validate_image(image)
    blurred = np.asarray(to_pil(x).filter(ImageFilter.GaussianBlur(2)), dtype=np.float32) / 255
    texture = np.abs(x - blurred)
    texture_image = Image.fromarray(np.rint(np.clip(texture * 4, 0, 1) * 255).astype(np.uint8))
    local_texture = np.asarray(texture_image.filter(ImageFilter.GaussianBlur(5)), dtype=np.float32) / (255 * 4)
    mask = local_texture > 0.008
    if np.count_nonzero(mask) < 64:
        raise ValueError('Insufficient textured foreground for this simulator; inspect source, do not silently substitute.')
    return mask

def case_seed(relative_path, family, severity, seed=20261110):
    key = f'{VERSION}|{seed}|{relative_path}|{family}|{severity}'
    return int.from_bytes(hashlib.sha256(key.encode('utf-8')).digest()[:8], 'big')

def degrade(image, family, severity, seed):
    if family not in ('dry_like', 'wet_like', 'generic') or severity not in SETTINGS:
        raise ValueError('Unknown degradation family or severity.')
    x = validate_image(image)
    fg = foreground_mask(x)
    settings = SETTINGS[severity]
    rng = np.random.default_rng(seed)
    source = to_pil(x)
    if family == 'generic':
        intervention = fg.astype(np.float32)
        blurred = np.asarray(source.filter(ImageFilter.GaussianBlur(settings['blur'])), dtype=np.float32) / 255
        candidate = 0.5 + settings['contrast'] * (blurred - 0.5)
        candidate = candidate + rng.normal(0, settings['noise'], size=x.shape)
    else:
        coarse = Image.fromarray(rng.integers(0, 256, size=(8, 8), dtype=np.uint8))
        field = np.asarray(coarse.resize(source.size, Image.Resampling.BICUBIC).filter(
            ImageFilter.GaussianBlur(5)), dtype=np.float32) / 255
        cutoff = np.quantile(field[fg], 1 - settings['coverage'])
        width = max(float(field[fg].std()) * 0.5, 1 / 255)
        intervention = np.clip((field - cutoff) / width + 0.5, 0, 1) * fg
        if family == 'dry_like':
            eroded = np.asarray(source.filter(ImageFilter.MaxFilter(settings['window'])), dtype=np.float32) / 255
            candidate = eroded + settings['strength'] * (1 - eroded)
        else:
            thickened = np.asarray(source.filter(ImageFilter.MinFilter(settings['window'])), dtype=np.float32) / 255
            candidate = thickened * (1 - 0.5 * settings['strength'])
    y = np.clip(x + intervention * (candidate - x), 0, 1).astype(np.float32)
    return y, intervention.astype(np.float32)
'''

simulator_bytes = SIMULATOR_SOURCE.encode('utf-8')
simulator_sha256 = hashlib.sha256(simulator_bytes).hexdigest()
module_path = WORK_DIR / 'fingerprint_degradation.py'
module_path.write_bytes(simulator_bytes)
spec = importlib.util.spec_from_file_location('fingerprint_degradation_v1', module_path)
if spec is None or spec.loader is None:
    raise RuntimeError('Cannot load simulator module.')
simulator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(simulator)
simulator_config = {
    'version': simulator.VERSION, 'module_sha256': simulator_sha256, 'seed': 20261110,
    'split_protocol_sha256': file_digest(SPLIT_DIR / 'protocol.json'),
    'normalization_configuration_sha256': file_digest(PAIR_DIR / 'configuration.json'),
    'settings': simulator.SETTINGS, 'families': ['dry_like', 'wet_like', 'generic'],
    'pillow_version': Image.__version__, 'numpy_version': np.__version__,
    'labels': 'synthetic image-domain heuristics, not observed skin moisture',
    'target': 'original impression, not guaranteed clean',
    'intervention_map': 'support of simulator operations, not calibrated reliability',
    'pilot_selection': 'first two sorted training scoped IDs per dataset/db; probe 2, gallery 1',
}
SIM_DIR.mkdir(parents=True, exist_ok=True)
frozen = {'fingerprint_degradation.py': simulator_bytes, 'configuration.json': json_bytes(simulator_config)}
for name, data in frozen.items():
    path = SIM_DIR / name
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f'Frozen simulator differs: {path}. Explicitly version any revision.')
for name, data in frozen.items():
    path = SIM_DIR / name
    if not path.exists():
        with path.open('xb') as output:
            output.write(data)
    if path.read_bytes() != data:
        raise RuntimeError(f'Simulator persistence failed: {path}')
print('Frozen simulator:', simulator_sha256)

In [ ]:
train = locked[locked['split'].eq('train')]
selected = []
for (dataset, db), group in train.groupby(['dataset', 'db'], sort=True):
    identities = sorted(group['identity_key'].unique())[:2]
    if len(identities) != 2 or dataset not in ('FVC2000', 'FVC2002'):
        raise RuntimeError('Expected two training fingers in each source database.')
    for identity in identities:
        finger = group[group['identity_key'].eq(identity)]
        probes = finger[finger['impression_id'].eq(2)]
        galleries = finger[finger['impression_id'].eq(1)]
        if len(probes) != 1 or len(galleries) != 1:
            raise RuntimeError(f'Missing or duplicate pilot impressions: {identity}')
        selected.append((probes.iloc[0].to_dict(), galleries.iloc[0].to_dict()))
if len(selected) != 12:
    raise RuntimeError('Expected 12 pilot training fingers across six databases.')

def load_normalized(record):
    source = DATA_ROOT / record['relative_path']
    if not source.is_file() or file_digest(source) != record['sha256']:
        raise RuntimeError(f'Missing or altered source: {source}. Restore with notebook 02 if needed.')
    dpi = normalization['source_dpi'][f"{record['dataset']}_{record['db']}"]
    with Image.open(source) as image:
        image = image.convert('L')
        if dpi != 500:
            image = image.resize(tuple(max(1, round(length * 500 / dpi)) for length in image.size),
                                 resample=Image.Resampling.LANCZOS)
        return np.asarray(image, dtype=np.float32) / 255

def save_image(array, path):
    simulator.to_pil(array).save(path, dpi=(500, 500))

cases = []
for probe, gallery in selected:
    identity = probe['identity_key']
    x = load_normalized(probe)
    gallery_image = load_normalized(gallery)
    case_dir = WORK_DIR / identity
    case_dir.mkdir(parents=True, exist_ok=True)
    target_path = case_dir / 'original_probe.png'
    gallery_path = case_dir / 'gallery.png'
    save_image(x, target_path)
    save_image(gallery_image, gallery_path)
    base = {
        'dataset': probe['dataset'], 'db': probe['db'], 'identity_key': identity,
        'source_relative_path': probe['relative_path'], 'source_sha256': probe['sha256'],
        'gallery_relative_path': gallery['relative_path'], 'gallery_sha256': gallery['sha256'],
        'target_path': str(target_path), 'gallery_path': str(gallery_path),
    }
    cases.append({**base, 'family': 'original', 'severity': 'none', 'seed': 'none',
                  'probe_path': str(target_path), 'mask_path': '', 'mae': 0.0,
                  'changed_fraction': 0.0, 'intervention_fraction': 0.0})
    for family in simulator_config['families']:
        for severity in simulator.SETTINGS:
            seed = simulator.case_seed(probe['relative_path'], family, severity, simulator_config['seed'])
            y, intervention = simulator.degrade(x, family, severity, seed)
            probe_path = case_dir / f'{family}_{severity}.png'
            mask_path = case_dir / f'{family}_{severity}_intervention.png'
            save_image(y, probe_path)
            save_image(intervention, mask_path)
            cases.append({**base, 'family': family, 'severity': severity, 'seed': str(seed),
                          'probe_path': str(probe_path), 'mask_path': str(mask_path),
                          'mae': float(np.abs(y - x).mean()),
                          'changed_fraction': float((np.abs(y - x) > 1 / 255).mean()),
                          'intervention_fraction': float((intervention > 0).mean())})
pilot = pd.DataFrame(cases)
if len(pilot) != 120 or not set(pilot['source_relative_path']).issubset(set(train['relative_path'])):
    raise RuntimeError('Unexpected pilot size or non-training source detected.')
pilot.to_csv(REPORT_DIR / 'pilot_cases.csv', index=False)
with zipfile.ZipFile(REPORT_DIR / 'pilot_images.zip', 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(WORK_DIR.rglob('*.png')):
        archive.write(path, arcname=path.relative_to(WORK_DIR).as_posix())
print(pilot.groupby(['family', 'severity'])[['mae', 'changed_fraction', 'intervention_fraction']].mean().to_string())
print('Generated 12 original + 108 degraded probes; images and intervention maps saved.')

In [ ]:
template_cache = {}

def template_for(path):
    path = Path(path)
    key = str(path)
    if key in template_cache:
        return template_cache[key]
    root = path.with_suffix('')
    result = subprocess.run([str(MINDTCT), str(path), str(root)],
                            capture_output=True, text=True, timeout=120)
    root.with_suffix('.log').write_text(result.stdout + result.stderr, encoding='utf-8')
    if result.returncode or not root.with_suffix('.xyt').is_file():
        raise RuntimeError(f'Pilot MINDTCT process failed: {path}\n{result.stderr}')
    points = []
    for line in root.with_suffix('.xyt').read_text(encoding='ascii').splitlines():
        if line.strip():
            values = [int(value) for value in line.split()]
            if len(values) != 4:
                raise RuntimeError(f'Malformed pilot minutiae: {path}')
            points.append(values)
    template_cache[key] = (root.with_suffix('.xyt'), len(points))
    return template_cache[key]

score_rows = []
for index, case in enumerate(pilot.to_dict('records'), start=1):
    probe_xyt, probe_count = template_for(case['probe_path'])
    gallery_xyt, gallery_count = template_for(case['gallery_path'])
    row = {**case, 'probe_minutiae': probe_count, 'gallery_minutiae': gallery_count,
           'matchable': probe_count >= 10 and gallery_count >= 10, 'score': np.nan}
    if row['matchable']:
        result = subprocess.run([str(BOZORTH3), '-A', 'minminutiae=10', str(probe_xyt), str(gallery_xyt)],
                                capture_output=True, text=True, timeout=120)
        if result.returncode or re.fullmatch(r'\s*\d+\s*', result.stdout) is None:
            raise RuntimeError(f'Pilot BOZORTH3 failed: {result.stdout}\n{result.stderr}')
        row['score'] = int(result.stdout.strip())
    score_rows.append(row)
    if index % 20 == 0:
        pd.DataFrame(score_rows).to_csv(REPORT_DIR / 'pilot_scores_progress.csv', index=False)
        print(f'Pilot matching {index}/{len(pilot)}', flush=True)
scored = pd.DataFrame(score_rows)
original_scores = scored[scored['family'].eq('original')].set_index('identity_key')['score']
scored['original_score'] = scored['identity_key'].map(original_scores)
scored['score_delta'] = scored['score'] - scored['original_score']
scored.to_csv(REPORT_DIR / 'pilot_scores.csv', index=False)
summary = scored.groupby(['family', 'severity']).agg(
    probes=('identity_key', 'size'), matchable=('matchable', 'sum'),
    comparable_deltas=('score_delta', 'count'), median_score_delta=('score_delta', 'median'),
    mean_mae=('mae', 'mean'),
).reset_index()
summary.to_csv(REPORT_DIR / 'pilot_summary.csv', index=False)
print(summary.to_string(index=False))
print('Missing scores are extraction failures, not zero similarity. Deltas use comparable cases only.')
print('No accuracy, EER or dry/wet-real claim is made from this small train pilot.')

In [ ]:
first_ids = pilot.groupby(['dataset', 'db'], sort=True)['identity_key'].first().tolist()
figure, axes = plt.subplots(6, 10, figsize=(20, 13))
for row_index, identity in enumerate(first_ids):
    examples = scored[scored['identity_key'].eq(identity)]
    for column, case in enumerate(examples.to_dict('records')):
        axis = axes[row_index, column]
        with Image.open(case['probe_path']) as image:
            axis.imshow(image, cmap='gray', vmin=0, vmax=255)
        value = f"{case['score']:.0f}" if case['matchable'] else 'FAIL'
        axis.set_title(f"{case['family']}\n{case['severity']} | score {value}", fontsize=7)
        axis.set_xticks([])
        axis.set_yticks([])
        if column == 0:
            axis.set_ylabel(identity, fontsize=7)
figure.tight_layout()
figure.savefig(REPORT_DIR / 'pilot_overview.png', dpi=140)
plt.show()
with zipfile.ZipFile(REPORT_DIR / 'pilot_templates_logs.zip', 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for extension in ('*.xyt', '*.log'):
        for path in sorted(WORK_DIR.rglob(extension)):
            archive.write(path, arcname=path.relative_to(WORK_DIR).as_posix())
completion = {
    'status': 'train_degradation_pilot_complete', 'run_id': RUN_ID,
    'simulator_configuration_sha256': file_digest(SIM_DIR / 'configuration.json'),
    'module_sha256': simulator_sha256, 'nbis_binary_sha256': build['binary_sha256'],
    'pilot_cases': len(scored), 'split': 'train', 'is_test_result': False,
    'report_hashes': {path.name: file_digest(path) for path in REPORT_DIR.iterdir() if path.is_file()},
}
(REPORT_DIR / 'completion.json').write_bytes(json_bytes(completion))
print('TRAIN DEGRADATION PILOT COMPLETE:', REPORT_DIR)
print('Review overview and summary before training. Next: a small restoration-only baseline, not GAN yet.')